In [1]:
import requests
import pandas as pd

def get_total_ids_from_geohash(g_hash="wy7e"):
    print(f"🚀 Geohash '{g_hash}' 구역에서 전체 ID 수집을 시작합니다...")
    url = f"https://apis.zigbang.com/house/property/v1/items/onerooms?geohash={g_hash}&depositMin=0&rentMin=0"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Origin": "https://www.zigbang.com",
        "Referer": "https://www.zigbang.com/"
    }
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"❌ 접속 에러: {response.status_code}")
            return []
        data = response.json()
        raw_items = data.get("items", [])
        collected_data = []
        for item in raw_items:
            collected_data.append({
                "id": item.get("id"),
                "lat": item.get("lat"),
                "lng": item.get("lng")
            })
        print(f"✅ 수집 완료! 총 {len(collected_data)}개의 아이템 정보를 확보했습니다.")
        return collected_data
    except Exception as e:
        print(f"❌ 에러 발생: {e}")
        return []

# 실행 및 저장
raw_data = get_total_ids_from_geohash()
if raw_data:
    df_raw = pd.DataFrame(raw_data)
    df_raw.to_csv("zigbang_raw_ids.csv", index=False)

🚀 Geohash 'wy7e' 구역에서 전체 ID 수집을 시작합니다...
✅ 수집 완료! 총 1222개의 아이템 정보를 확보했습니다.


In [ ]:
def filter_yeongnam_core_items(input_file="zigbang_raw_ids.csv", output_file="zigbang_filtering.csv"):
    print(f"🧹 '{input_file}' 파일에서 고산역까지의 핵심 매물 필터링을 시작합니다...")
    try:
        df = pd.read_csv(input_file)
        FIXED_BOUNDS = {
            'min_lat': 35.7965, # 계산된 왼쪽 아래 위도
            'max_lat': 35.8479, # 오른쪽 위 위도
            'min_lon': 128.6842, # 계산된 왼쪽 아래 경도
            'max_lon': 128.7682  # 오른쪽 위 경도
        }
        mask = (
            (df['lat'] >= FIXED_BOUNDS['min_lat']) & (df['lat'] <= FIXED_BOUNDS['max_lat']) &
            (df['lng'] >= FIXED_BOUNDS['min_lon']) & (df['lng'] <= FIXED_BOUNDS['max_lon'])
        )
        filtered_df = df[mask].copy()
        filtered_df = filtered_df.drop_duplicates(subset=['id'])
        filtered_df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"✅ 필터링 완료! {len(df)}개 -> {len(filtered_df)}개 저장됨")
    except Exception as e:
        print(f"❌ 에러 발생: {e}")

# 실행
filter_yeongnam_core_items()

🧹 'zigbang_raw_ids.csv' 파일에서 영남대역 핵심 매물 필터링을 시작합니다...
✅ 필터링 완료! 1222개 -> 1197개 저장됨


In [3]:
import time
import pandas as pd
import requests
from datetime import datetime

def collect_perfect_details_v3():
    print("🚀 매물 전수 조사를 시작합니다. (V3 정밀 모드)")
    
    # 1. 현재 연도 정의 (반드시 필요)
    current_year = datetime.now().year
    
    try:
        # 파일 경로가 맞는지 다시 한번 확인하세요!
        df_base = pd.read_csv("zigbang_filtering.csv")
        item_ids = df_base['id'].astype(str).tolist()
    except Exception as e:
        print(f"❌ 로드 실패: {e}"); return

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    final_results = []
    
    for idx, item_id in enumerate(item_ids, 1):
        url = f"https://apis.zigbang.com/v3/items/{item_id}?version=&domain=zigbang"
        try:
            response = requests.get(url, headers=headers)
            if response.status_code == 200:
                data = response.json()
                item = data.get("item", {})
                price = item.get("price", {})
                loc = item.get("location", {})
                area = item.get("area", {})
                floor = item.get("floor", {})
                options = item.get("options", [])
                
                # 노후도 계산
                approve_date = item.get("approveDate", "")
                age = None
                if approve_date and len(approve_date) >= 2:
                    try:
                        year_prefix = int(approve_date[:2])
                        build_year = 2000 + year_prefix if year_prefix < 50 else 1900 + year_prefix
                        age = current_year - build_year
                    except:
                        age = None

                res = {
                    "매물번호": item_id,
                    "보증금": price.get("deposit"),
                    "월세": price.get("rent"),
                    "관리비": item.get("manageCost", {}).get("amount"),
                    "전용면적": area.get("전용면적M2"),
                    "노후도": age,
                    "해당층": floor.get("floor"),
                    "전체층": floor.get("allFloors"),
                    "엘리베이터": item.get("elevator"),
                    "방향": item.get("roomDirection"),
                    "위도": loc.get("lat"),
                    "경도": loc.get("lng"),
                    "주소": item.get("addressOrigin", {}).get("fullText", "").strip(),
                    "옵션_냉장고": "냉장고" in options,
                    "옵션_침대": "침대" in options,
                    "옵션_전자레인지": "전자레인지" in options,
                    "옵션_에어컨": "에어컨" in options,
                    "옵션_세탁기": "세탁기" in options
                }

                pois = item.get("neighborhoods", {}).get("nearbyPois", [])
                for poi in pois:
                    if poi.get("exists"):
                        res[f"{poi.get('poiType')}_거리(m)"] = poi.get("distance")
                
                final_results.append(res)

                if idx % 50 == 0: 
                    print(f"✅ {idx}/{len(item_ids)} 완료...")
                    # 중간 저장 (선택 사항)
                    # pd.DataFrame(final_results).to_csv("zigbang_backup.csv", index=False, encoding="utf-8-sig")
            
            time.sleep(0.5)
            
        except Exception as e:
            print(f"❌ 에러 발생 (매물번호 {item_id}): {e}")

    # 최종 저장
    if final_results:
        df_final = pd.DataFrame(final_results)
        df_final.to_csv("zigbang_dup.csv", index=False, encoding="utf-8-sig")
        print(f"✨ 상세 수집 완료! 총 {len(df_final)}건 저장됨.")
    else:
        print("⚠ 수집된 데이터가 없습니다.")

# 실행
collect_perfect_details_v3()

🚀 매물 전수 조사를 시작합니다. (V3 정밀 모드)
✅ 50/1197 완료...
✅ 100/1197 완료...
✅ 150/1197 완료...
✅ 200/1197 완료...
✅ 250/1197 완료...
✅ 300/1197 완료...
✅ 350/1197 완료...
✅ 400/1197 완료...
✅ 450/1197 완료...
✅ 500/1197 완료...
✅ 550/1197 완료...
✅ 600/1197 완료...
✅ 650/1197 완료...
✅ 700/1197 완료...
✅ 750/1197 완료...
✅ 800/1197 완료...
✅ 850/1197 완료...
✅ 900/1197 완료...
✅ 950/1197 완료...
✅ 1000/1197 완료...
✅ 1050/1197 완료...
✅ 1100/1197 완료...
✅ 1150/1197 완료...
✨ 상세 수집 완료! 총 1197건 저장됨.


In [6]:
def finalize_data(input_file='zigbang_dup.csv', output_file='zigbang.csv'):
    try:
        df = pd.read_csv(input_file)
        
        # 중복 좌표 제거
        initial_count = len(df)
        df_final = df.drop_duplicates(subset=['위도', '경도', '해당층', '보증금', '월세'], keep='first')
        
        df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"📊 최종 결과: {initial_count}건 -> {len(df_final)}건 (중복 제거 완료)")
        print(f"💾 '{output_file}' 저장 완료.")
    except Exception as e:
        print(f"❌ 에러 발생: {e}")

# 실행
finalize_data()

📊 최종 결과: 1197건 -> 983건 (중복 제거 완료)
💾 'zigbang.csv' 저장 완료.


In [ ]:
import pandas as pd

# 1. 파일 로드 (이전 단계에서 저장한 zigbang.csv를 불러옵니다)
df = pd.read_csv("zigbang.csv")

df['해당층'] = df['해당층'].replace('반지하', 0)

df['월세'] = df['월세'] + df['관리비'].fillna(0)

# 3. 남향 여부 추출 (방향 컬럼에 'S'가 포함되어 있으면 True)
# 'S', 'SE', 'SW' 등을 모두 남향 계열로 간주합니다.
df['남향'] = df['방향'].fillna('').str.contains('S', case=False)

# 4. 풀옵션 여부 판단
# 요청하신 5가지 옵션이 모두 True인 경우만 '풀옵션' 컬럼을 True로 설정합니다.
option_cols = ['옵션_냉장고', '옵션_침대', '옵션_전자레인지', '옵션_에어컨', '옵션_세탁기']
df['풀옵션'] = df[option_cols].all(axis=1)

# 5. 불필요한 컬럼 삭제
# 명시하신 '매물번호', '주소'와 이미 통합/변환이 완료된 '관리비', '방향'도 함께 제거하여 깔끔하게 만듭니다.
drop_cols = ['매물번호', '주소', '관리비', '방향', '옵션_냉장고', '옵션_침대', '옵션_전자레인지', '옵션_에어컨', '옵션_세탁기']
df_processed = df.drop(columns=drop_cols)

# 6. 전처리 결과 확인 및 저장
print(f"✅ 전처리 완료! (현재 컬럼 수: {len(df_processed.columns)}개)")
print("-" * 30)
print(df_processed[['월세', '남향', '풀옵션']].head()) # 주요 변경 항목 확인
df_processed['해당층'] = pd.to_numeric(df_processed['해당층'], errors='coerce').fillna(0).astype(int)


# 분석용 최종 파일 저장
df_processed.to_csv("zigbang_processed_option.csv", index=False, encoding="utf-8-sig")


print("-" * 30)
print("💾 'zigbang_processed.csv'로 저장되었습니다. 이제 머신러닝을 시작할 수 있습니다!")

✅ 전처리 완료! (현재 컬럼 수: 22개)
------------------------------
     월세     남향
0  35.0   True
1  35.0  False
2  37.0  False
3  50.0  False
4  32.0  False
------------------------------
💾 'zigbang_processed.csv'로 저장되었습니다. 이제 머신러닝을 시작할 수 있습니다!


In [12]:
df = pd.read_csv('zigbang_processed.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 983 entries, 0 to 982
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   보증금          983 non-null    int64  
 1   월세           983 non-null    float64
 2   전용면적         983 non-null    float64
 3   노후도          978 non-null    float64
 4   해당층          983 non-null    int64  
 5   전체층          983 non-null    int64  
 6   엘리베이터        983 non-null    bool   
 7   위도           983 non-null    float64
 8   경도           983 non-null    float64
 9   지하철역_거리(m)   982 non-null    float64
 10  세탁소_거리(m)    982 non-null    float64
 11  카페_거리(m)     982 non-null    float64
 12  약국_거리(m)     982 non-null    float64
 13  대형마트_거리(m)   982 non-null    float64
 14  편의점_거리(m)    982 non-null    float64
 15  버스정류장_거리(m)  982 non-null    float64
 16  남향           983 non-null    bool   
 17  풀옵션          983 non-null    bool   
dtypes: bool(3), float64(12), int64(3)
memory usage: 11

In [13]:
df = pd.read_csv("zigbang_processed.csv")

# 2. 결측치 처리 (요청하신 규칙)
# 노후도: 평균으로 채우기
df['노후도'] = df['노후도'].fillna(df['노후도'].mean())

# 거리 관련 컬럼들 리스트
infra_cols = ['지하철역_거리(m)', '세탁소_거리(m)', '카페_거리(m)', 
              '약국_거리(m)', '대형마트_거리(m)', '편의점_거리(m)', '버스정류장_거리(m)']

# 거리 컬럼에 결측치가 있는 행은 삭제
df = df.dropna(subset=infra_cols)

# 3. 이상치 판단 및 사유 기록 초기화
cols_to_check = ['월세', '보증금', '전용면적']
df['is_outlier'] = False    # 이상치 여부 플래그
df['outlier_reason'] = ""   # 이상치 사유를 기록할 컬럼

for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # 하한선 미달 체크 및 사유 추가
    low_mask = df[col] < lower
    df.loc[low_mask, 'outlier_reason'] += f"{col}_low "
    
    # 상한선 초과 체크 및 사유 추가
    high_mask = df[col] > upper
    df.loc[high_mask, 'outlier_reason'] += f"{col}_high "
    
    # 해당 컬럼에서 하나라도 범위를 벗어나면 is_outlier를 True로 변경
    df.loc[low_mask | high_mask, 'is_outlier'] = True

# 4. 사유 컬럼 정제 (앞뒤 공백 제거 및 빈 값을 'None'으로 변경)
df['outlier_reason'] = df['outlier_reason'].str.strip()
df.loc[df['outlier_reason'] == "", 'outlier_reason'] = "None"

# 5. 결과 저장
df.to_csv("zigbang_outlier.csv", index=False, encoding="utf-8-sig")

print(f"✅ 전체 {len(df)}건 중 이상치 {df['is_outlier'].sum()}건 표시 및 사유 기록 완료!")
# 이상치인 데이터만 슬쩍 확인해보기
print(df[df['is_outlier'] == True][['보증금', '월세', '전용면적', 'outlier_reason']].head())

✅ 전체 982건 중 이상치 139건 표시 및 사유 기록 완료!
    보증금    월세   전용면적    outlier_reason
3   300  50.0  30.00           월세_high
6   200  35.0  49.59         전용면적_high
9   500  53.0  16.82  월세_high 보증금_high
15  300  50.0  30.00           월세_high
16  300  50.0  36.36           월세_high
